# 🦾 Training Toolkit: Fine-tune PaliGemma for instance segmentation

- **Input**: image, text prompt saying `segment {class 1} ; {class 2} ...`
- **Output**: bounding boxes and segmentation masks for objects on the image

In [ ]:
from dotenv import load_dotenv  
from pathlib import Path
import sys

sys.path.append(Path("..").resolve().as_posix())  # add the root of the project to the path to enable imports
_ = load_dotenv() # load HF token to access PaliGemma's gated repo

## Easy mode 🙈

**To get started**, convert your dataset to HF 🤗 Datasets format and save it locally.

It needs to have the following columns:

- `image` column that is cast to `datasets.Image()` (see HF docs)
- `prompt` column that contains an input prompt `segment {class 1} ; {class 2} ...`
- `xyxy_bboxes`: list of numpy arrays with `[x1, y1, x2, y2]` bounding box coordinates for every object on the image
- `masks`: list of `bool` numpy matrices, each representing a pixel-wise mask for the object
- `classes`: list of class names corresponding to bboxes and masks.

In [ ]:
DATASET_PATH = "path/to/dataset"

In [ ]:
from training_toolkit import build_trainer, paligemma_image_preset, image_segmentation_preset

In [ ]:
trainer = build_trainer(
    **paligemma_image_preset.as_kwargs(),
    **image_segmentation_preset.with_path(DATASET_PATH).as_kwargs(),
)

In [ ]:
trainer.train()

## Advanced mode 🙉

Preset parameters don't quite fit your case? Doesn't mean you need to build *everything* from scratch.

Training PaliGemma to perform segmentation is no joke. Many bespoke parts need to be put in place for it to work. Learn how to combine Training Toolkit with custom tools to minimize time and energy required to train your adapter.

### 1. Tune preset parameters

Both `paligemma_image_preset` and `image_segmentation_preset` are Pydantic dataclasses.

- Both return a dict when you call `as_kwargs()` on them.
- Combined, those dicts contain all parameters necessary to train an adapter.
- You can edit those parameters directly as dataclass attributes.

In [ ]:
from training_toolkit import build_trainer, paligemma_image_preset, image_segmentation_preset

In [ ]:
# let's take a look at the arguments build_trainer expects

?build_trainer

In [ ]:
# we can find all these argumets in the presets

print(f"model preset: {paligemma_image_preset.as_kwargs().keys()}")
print(f"data preset args: {image_segmentation_preset.with_path("path/to/dataset").as_kwargs().keys()}")

In [ ]:
# let's edit a couple hyperparameters 

paligemma_image_preset.hf_model_id="google/paligemma-3b-mix-448"

paligemma_image_preset.training_args["per_device_train_batch_size"] = 8
paligemma_image_preset.training_args["per_device_eval_batch_size"] = 8
paligemma_image_preset.training_args["eval_strategy"] = "no"
paligemma_image_preset.training_args["num_train_epochs"] = 10

In [ ]:
# now as_kwargs will return updated arguments

paligemma_image_preset.as_kwargs()

### 2. Customize dataset format

Say your dataset doesn't match the strict criteria described above. Or you have your own idea of what exactly the inputs and the targets should look like.

In order to use your dataset with the rest of the toolkit, you need to write **your own data collator**. This is a utility that turns a list of individual samples from your dataset into a batch ready to go into the model. Those batches are complete with padded tokenized text, attention masks and preprocessed images.

The HF 🤗 Transformers processor (see HF docs) is going to do the conversion. We need to do the following:

1. Pluck images out of the dataset
2. Prepare text inputs (aka prompts or prefixes)
3. Convert segmentations into special text tokens. `SegmentationTokenizer` will take care of that. These are going to become suffixes - bits that the model will learn to generate.
4. Feed all those things into the processor
5. Return the result

In [ ]:
from training_toolkit import DataPreset
from training_toolkit.common.tokenization_utils.segmentation import (
    SegmentationTokenizer,
)


class SpecialImageSegmentationCollator:

    def __init__(self, processor):
        self.processor = processor
        self.segmentation_tokenizer = SegmentationTokenizer()

    def __call__(self, examples):
        # 1. gather images into a list
        images = [example["image"] for example in examples]
        # 2. gather premade text inputs
        texts = [example["prompt"] for example in examples]

        # 3. convert segmentations into text token sequences
        labels = [
            self.segmentation_tokenizer.encode(
                example["image"],
                example["xyxy_bboxes"],
                example["masks"],
                example["classes"],
            )
            for example in examples
        ]

        batch = self.processor(
            text=texts,
            images=images,
            suffix=labels,
            return_tensors="pt",
            padding="longest",
        )
        return batch


# let's wrap the new collator into a custom DataPreset
special_image_segmentation_preset = DataPreset(
    train_test_split=0.1,
    collator_cls=SpecialImageSegmentationCollator,
)

# finally, we can make sure everything loads correctly
special_image_segmentation_preset.with_path("path/to/data").as_kwargs()

### 3. Convert a COCO dataset

Let's walk through the process of conveting one of the nastier dataset formats into something that can work with the Training Toolkit.

Our goal is to convert the dataset into an HF 🤗 Datasets dataset with a specific layout.

In [ ]:
from datasets import Dataset, Image
from collections import defaultdict
import PIL

from tqdm import tqdm
from pycocotools.coco import COCO
import numpy as np
import cv2
import albumentations as A

from pathlib import Path

In [ ]:
dataset_path = Path("path/to/data/")
coco = COCO(dataset_path.joinpath("annotations.json").as_posix())

image_ids = coco.getImgIds()

# get a list of class names
class_names = [coco.cats[cat_id]["name"] for cat_id in coco.getCatIds()]

In [ ]:
IMAGE_SIZE = 512
LIMIT_SAMPLES = 500

# HF Datasets may choke on full images, so we'll resize them to a smaller size
# We are using Albumentations to crop and resize images together with their annotations

transform = A.Compose(
    [
        A.SmallestMaxSize(max_size=IMAGE_SIZE, always_apply=True),
        A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, always_apply=True),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc", label_fields=["class_labels"], clip=True, min_area=1
    ),
)

dataset_dict = defaultdict(list)
prefix = "segment " + " ; ".join(class_names)  # this is the prompt for the model

for image_id in tqdm(image_ids):

    # 1. Parse COCO annotations
    image_path = dataset_path.joinpath(coco.loadImgs(image_id)[0]["file_name"])
    annotations = coco.loadAnns(coco.getAnnIds(image_id))
    xywh_bboxes = [ann["bbox"] for ann in annotations]
    xyxy_bboxes = [[x, y, x + w, y + h] for x, y, w, h in xywh_bboxes]
    classes = [class_names[ann["category_id"]] for ann in annotations]

    # 2. Load and resize the image and its annotations
    image = cv2.imread(image_path.as_posix())
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    masks = [coco.annToMask(ann) for ann in annotations]

    transformed = transform(
        image=image, masks=masks, bboxes=xyxy_bboxes, class_labels=classes
    )

    # 3. Prepare the sample for storage
    image = PIL.Image.fromarray(transformed["image"])
    masks = np.array(transformed["masks"], dtype=bool)
    xyxy_bboxes = np.array(transformed["bboxes"], dtype=int)
    classes = transformed["class_labels"]

    xyxy_bboxes = np.array(
        [
            [x1, y1, x2, y2]
            for x1, y1, x2, y2 in xyxy_bboxes
            if x2 - x1 > 0 and y2 - y1 > 0
        ]
    )

    if len(masks) == 0 or len(xyxy_bboxes) == 0 or len(classes) == 0:
        continue

    assert len(masks.shape) == 3
    assert (
        len(xyxy_bboxes.shape) == 2 and xyxy_bboxes.shape[1] == 4
    ), f"{xyxy_bboxes.shape}, {len(masks)}, {len(xyxy_bboxes)}"

    # 4. Store the sample
    dataset_dict["image"].append(image)
    dataset_dict["prompt"].append(prefix)
    dataset_dict["xyxy_bboxes"].append(xyxy_bboxes)
    dataset_dict["masks"].append(masks)
    dataset_dict["classes"].append(classes)


# Convert the dataset to HF format and save it to disk
dataset = Dataset.from_dict(dataset_dict)
dataset = dataset.cast_column("image", Image())

dataset.info.dataset_name = "special_dataset"
dataset.info.description = f"class_names: {' ; '.join(class_names)}"

dataset.save_to_disk("special_dataset")

### 4. Use SegmentationTokenizer separately

`SegmentationTokenizer` wraps a special pretrained autoencoder that enables the whole "segmentation through LLM" business. It converts segmentation masks into sequences of 20 tokens (and back). This is a handy tool for both training and inference. 

In [ ]:
from datasets import Dataset
import PIL
import numpy as np
from training_toolkit.common.tokenization_utils.segmentation import (
    SegmentationTokenizer,
)

In [ ]:
dataset = Dataset.load_from_disk("path/to/dataset")
dataset = dataset.with_format("torch")

segmentation_tokenizer = SegmentationTokenizer()

In [ ]:
# 1. Let's take a look at the original image
example = dataset[0]
PIL.Image.fromarray(example["image"].permute(1, 2, 0).numpy())

In [ ]:
# 2. ...and it's mask
PIL.Image.fromarray(example["masks"][0].numpy())

In [ ]:
# 3. Now let's encode the mask and take a look at the resulting token

suffix = segmentation_tokenizer.encode(
    example["image"], example["xyxy_bboxes"], example["masks"], example["classes"]
)

suffix

In [ ]:
# 4. Finally, let's decode the token sequence back into a pixel-level mask again
decoded = segmentation_tokenizer.decode(suffix, 512, 512)

PIL.Image.fromarray((decoded[0]["mask"] > 0.5).astype(np.uint8) * 255)